# Data Generation
This notebook generates synthetic data for a fictional B2B SaaS company, BlueTide, and its customer subscription base. The generated datasets will later be imported into Snowflake for cleaning, transformation, and analysis using SQL.

There are five core tables:
1. Companies – Customer accounts and firmographics
2. Users/Licenses – Seats and user activity
3. Subscriptions – Plans, pricing, and billing status
4. Product Usage – Platform engagement metrics
5. Support Tickets – Customer support interactions

The primary goal of this project is to analyze Customer Churn. For this analysis, the business currently associates churn risk with:
- Declining platform usage or license counts
- Missed or delayed payments
- Increased (or absent) support activity

## Business Context (The Story)

During an initial conversation with the BlueTide Customer Success team, several team members reported that "Some of our larger customers seem less active, and a few have already left.”

The follow-up questions included 
- What does “leaving” mean? (Full cancellation, downgrades, seat reductions?)
- How long has this been happening?
- Which customer segments are most affected?

To which the response was, "We’re not sure. We just know it’s happening.”

This led to the core analytical question, **“What is driving churn, and how can we identify it early?”**

This notebook represents the first step in answering that question by creating a realistic data environment for analysis. 

## Data Model
This section outlines the structure of the datasets generated in this project. These tables are designed to simulate a B2B DevOps SaaS environment and support downstream churn analysis.

### Companies
*Account-level customer information*

| Column       | Description                          |
| ------------ | ------------------------------------ |
| company_id   | Unique company identifier            |
| company_name | Company name                         |
| industry     | Industry classification              |
| company_size | Number of employees                  |
| country      | Company country                      |
| state        | Company state                        |
| signup_date  | Account creation date                |
| plan_type    | Current subscription plan            |
| status       | Active / Churned / Downgraded        |
| churn_date   | Date of cancellation (if applicable) |

### Users/Licenses
*Licensed users and seat activity*

| Column           | Description                          |
| ---------------- | ------------------------------------ |
| user_id          | Unique user identifier               |
| company_id       | Associated company                   |
| company_state    | State where company is based         |
| role             | User role (Admin, Developer, Viewer) |
| join_date        | Date user joined                     |
| last_active_date | Most recent activity                 |
| is_active        | Active license flag                  |


### Subscriptions
*Billing and contract details*

| Column          | Description                     |
| --------------- | ------------------------------- |
| subscription_id | Unique subscription ID          |
| company_id      | Associated company              |
| plan_name       | Subscription tier               |
| start_date      | Subscription start              |
| end_date        | Subscription end (if cancelled) |
| price_per_user  | Monthly cost per seat           |
| user_count      | Number of licenses              |
| auto_renew      | Auto-renewal flag               |
| payment_status  | Paid / Late / Failed            |
| monthly_revenue | Calculated MRR                  |

### Product Usage
*Platform engagement metrics over time*

| Column         | Description            |
| -------------- | ---------------------- |
| usage_id       | Unique usage record ID |
| company_id     | Associated company     |
| usage_date     | Activity date          |
| active_users   | Daily active users     |
| commits        | Code commits           |
| pipelines_run  | CI/CD executions       |
| deployments    | Production deployments |
| merge_requests | Merge requests created |


### Support Tickets
*Customer support and incident data*

| Column             | Description                      |
| ------------------ | -------------------------------- |
| ticket_id          | Unique ticket ID                 |
| company_id         | Associated company               |
| created_date       | Ticket creation date             |
| resolved_date      | Resolution date                  |
| priority           | Low / Medium / High / Critical   |
| category           | Bug / Billing / Feature / Outage |
| status             | Open / Resolved / Escalated      |
| satisfaction_score | Post-resolution rating           |


**Table Relationships**
- `companies.company_id` is the primary key
- `company_id` is the foreign key in all other tables

All behavioral, revenue, and support data is linked at the company level

## Imports
This section contains the Python libraries required for data generation and export.

In [1]:
# core libraries
import pandas as pd
import numpy as np

# data generation
!pip install faker 
from faker import Faker
import random
from datetime import datetime, timedelta

# for exporting the final csvs
from IPython.display import FileLink

### Faker and Random Initialization

In [2]:
fake = Faker()

# ensure reproducibility
np.random.seed(42)
random.seed(42)

### Generate Companies

In [3]:
# set the number of companies and ID range
num_companies = 500  
company_ids = range(1, num_companies+1)

# define the states
us_states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
]

# create the companies dataframe
companies = pd.DataFrame({ 'company_id': company_ids
                         , 'company_name': [fake.company() for _ in company_ids]
                         , 'industry': [random.choice(['Software', 'FinTech', 'Healthcare', 'E-commerce']) for _ in company_ids]
                         , 'company_size': [random.choice([10, 25, 50, 100, 250, 500, 1000]) for _ in company_ids]
                         , 'country': ['US' for _ in company_ids]  # localize to US
                         , 'state': [random.choice(us_states) for _ in company_ids]  
                         , 'signup_date': [fake.date_between(start_date='-5y', end_date='today') for _ in company_ids]
                         , 'plan_type': [random.choice(['Free', 'Pro', 'Team', 'Enterprise']) for _ in company_ids]
                         , 'status': ['Active' for _ in company_ids]  
                         , 'churn_date': [pd.NaT for _ in company_ids]
                         })

### Generate Users/Licesnses

In [4]:
users = []
user_id = 1

# loop through each company
for company in companies.itertuples():
    
    # estimate users based on company size
    num_users = max(1, company.company_size // 10)

    ## add details for each user
    for _ in range(num_users):
        join_date = fake.date_between(company.signup_date, 'today')
        last_active = fake.date_between(join_date, 'today')

        users.append({  'user_id': user_id
                      , 'company_id': company.company_id
                      , 'company_state': company.state
                      , 'role': random.choice(['Admin', 'Developer', 'Viewer'])
                      , 'join_date': join_date
                      , 'last_active_date': last_active
                      , 'is_active': True
                     })

        user_id += 1

users_df = pd.DataFrame(users)

### Generate Subscriptions

In [5]:
subscriptions = []

for company in companies.itertuples():
    subscriptions.append({
        'subscription_id': company.company_id,
        'company_id': company.company_id,
        'plan_name': company.plan_type,
        'start_date': company.signup_date,
        'end_date': pd.NaT,  # simulate churn later
        'price_per_user': random.choice([0, 29, 59, 99]),
        'user_count': len(users_df[users_df.company_id == company.company_id]),
        'auto_renew': random.choice([True, False]),
        'payment_status': random.choice(['Paid', 'Late']),
        'monthly_revenue': 0  # calculate as price_per_user * user_count
    })

subscriptions_df = pd.DataFrame(subscriptions)
subscriptions_df['monthly_revenue'] = subscriptions_df['price_per_user'] * subscriptions_df['user_count']

### Generate Product Usage

In [6]:
product_usage = []

for company in companies.itertuples():
    for i in range(90):  # last 3 months
        date = datetime.today() - timedelta(days=i)
        product_usage.append({
            'usage_id': f"{company.company_id}_{i}",
            'company_id': company.company_id,
            'usage_date': date,
            'active_users': random.randint(0, len(users_df[users_df.company_id == company.company_id])),
            'commits': random.randint(0, 50),
            'pipelines_run': random.randint(0, 20),
            'deployments': random.randint(0, 5),
            'merge_requests': random.randint(0, 15)
        })

product_usage_df = pd.DataFrame(product_usage)

### Generate Support Tickets

In [7]:
tickets = []

for company in companies.itertuples():
    num_tickets = random.randint(0, 10)
    for i in range(num_tickets):
        created = fake.date_between(company.signup_date, 'today')
        resolved = created + timedelta(days=random.randint(0,5))
        tickets.append({
            'ticket_id': f"{company.company_id}_{i}",
            'company_id': company.company_id,
            'created_date': created,
            'resolved_date': resolved,
            'priority': random.choice(['Low','Medium','High','Critical']),
            'category': random.choice(['Bug','Billing','Feature','Outage']),
            'status': random.choice(['Open','Resolved','Escalated']),
            'satisfaction_score': random.randint(1,5)
        })

tickets_df = pd.DataFrame(tickets)

## Simulating Churm
In this section, randomized churn is added to simulate companies "quiet quitting" the software. 

### Set Churn Customers Randomly
20% of Customers are set at random to display signs of churn. 

In [8]:
# set the churn rate
churn_rate = 0.2  # 20%

# get a random sample of the customers and apply a churn rate
churn_companies = companies.sample(
    frac=churn_rate,
    random_state=42
)['company_id'].tolist()

### Add the designation to Companies

In [9]:
# set today's date/time
today = pd.Timestamp.today()

# add churn date to the designated companies
for i, row in companies.iterrows():
    if row['company_id'] in churn_companies:
        churn_date = today - pd.Timedelta(days=random.randint(30,180))

        companies.at[i,'status'] = 'Churned'
        companies.at[i,'churn_date'] = churn_date

### Reduce Users
Reduce the number of active licenses and user activity. 

In [10]:
# loop through users
for i, row in users_df.iterrows():
    # loop through companies
    if row['company_id'] in churn_companies:
        if random.random() < 0.6:  # 60% go inactive
            inactive_date = companies.loc[companies.company_id == row.company_id,'churn_date'].values[0] - pd.Timedelta(days=random.randint(15,60))
            
            users_df.at[i,'last_active_date'] = inactive_date
            users_df.at[i,'is_active'] = False

### Reduce Product Usage

In [11]:
# loop through the churn companies' product usage
for i, row in product_usage_df.iterrows():
    
    if row['company_id'] in churn_companies:
        
        product_usage_df.at[i,'active_users'] = int(row['active_users'] * random.uniform(0.2, 0.5))
        product_usage_df.at[i,'commits'] = int(row['commits'] * random.uniform(0.3, 0.6))
        product_usage_df.at[i,'pipelines_run'] = int(row['pipelines_run'] * random.uniform(0.3, 0.6))

### Simulate Late Payments

In [12]:
# loop through subscriptions
for i, row in subscriptions_df.iterrows():

    # for churn customers
    if row['company_id'] in churn_companies:

        # set their payment status to late or failed, turn off autopay
        subscriptions_df.at[i,'payment_status'] = random.choice(['Late','Failed'])
        subscriptions_df.at[i,'auto_renew'] = False

### Increase Support Tickets

In [13]:
# loop through tickets
for i, row in tickets_df.iterrows():
    # add high and critical ticket designations
    if row['company_id'] in churn_companies:
        
        tickets_df.at[i,'priority'] = random.choice(['High','Critical'])
        tickets_df.at[i,'satisfaction_score'] = random.randint(1,3)

## Data Validation & Summary

This section performs basic validation checks and produces summary statistics to confirm that the synthetic data is realistic and consistent. 

### Company-Level

In [14]:
print("Total Companies:", len(companies))
print("\nCompany Status Breakdown:")
print(companies['status'].value_counts())

print("\nPlans Distribution:")
print(companies['plan_type'].value_counts())

print("\nCompanies by State (Top 10):")
print(companies['state'].value_counts().head(10))

Total Companies: 500

Company Status Breakdown:
status
Active     400
Churned    100
Name: count, dtype: int64

Plans Distribution:
plan_type
Team          139
Free          123
Enterprise    121
Pro           117
Name: count, dtype: int64

Companies by State (Top 10):
state
CT    17
UT    16
MS    14
NJ    14
GA    14
WV    14
PA    13
FL    13
NE    13
TX    13
Name: count, dtype: int64


**Checks**
1. We have churned customers.
2. The mix of plans is reasonable.
3. The State distribution is realistic.
   
   
### Users/Licenses

In [15]:
print("Total Users:", len(users_df))

print("\nActive vs Inactive Users:")
print(users_df['is_active'].value_counts())

print("\nAverage Users per Company:")
print(
    users_df.groupby('company_id')['user_id']
    .count()
    .mean()
    .round(2)
)

Total Users: 13562

Active vs Inactive Users:
is_active
True     11748
False     1814
Name: count, dtype: int64

Average Users per Company:
27.12


**Checks**
1. Not all users are active.
2. There are no empty companies.
3. The average seat size tracks.
   
   
### Subscriptions

In [16]:
print("Total Monthly Revenue:",
      round(subscriptions_df['monthly_revenue'].sum(), 2))

print("\nPayment Status:")
print(subscriptions_df['payment_status'].value_counts())

print("\nAverage Revenue per Company:",
      round(subscriptions_df['monthly_revenue'].mean(), 2))

Total Monthly Revenue: 728822

Payment Status:
payment_status
Late      257
Paid      193
Failed     50
Name: count, dtype: int64

Average Revenue per Company: 1457.64


**Checks**
1. Revenue is more thatn $0.
2. There are late/failed payments but the skew is not drastic.
   

### Product Usage

In [17]:

usage_summary = product_usage_df.groupby('company_id').agg({
    'active_users': 'mean',
    'commits': 'mean',
    'pipelines_run': 'mean',
    'deployments': 'mean'
}).round(2)

usage_summary.head()

,active_users,commits,pipelines_run,deployments
company_id,,,,
1,0.00,11.72,3.81,2.58
2,2.56,26.17,9.63,2.47
3,1.36,10.61,4.27,2.46
4,2.66,27.26,10.13,2.52
5,0.87,26.13,9.88,2.43


**Checks**
1. There are fewet churn than healthy customers. 
2. Not all have 0 usage.
3. The ranges are reasonable.


### Support Tickets

In [18]:
print("Total Tickets:", len(tickets_df))

print("\nTicket Priority:")
print(tickets_df['priority'].value_counts())

print("\nAverage Satisfaction Score:",
      round(tickets_df['satisfaction_score'].mean(), 2))

Total Tickets: 2502

Ticket Priority:
priority
High        791
Critical    762
Medium      490
Low         459
Name: count, dtype: int64

Average Satisfaction Score: 2.8


**Checks**
1. Some tickets are high priority.
2. Some are low satisfaction.
3. There are no perfect scores. 

### Relationship Validation

In [19]:
print("Companies without users:",
      companies[~companies.company_id.isin(users_df.company_id)].shape[0])

print("Companies without subscriptions:",
      companies[~companies.company_id.isin(subscriptions_df.company_id)].shape[0])

Companies without users: 0
Companies without subscriptions: 0


**Checks**   

All companies have users and subscriptions. 

## Export

The final data is comprised of
- 500 companies (400 Active, 100 Churned)
- Plan distribution: Enterprise > Team > Pro > Free
- Customers localized to the US with realistic state spread
- Users, subscriptions, product usage, and support tickets all linked and internally consistent
- Churn signals embedded: declining usage, inactive users, late payments, high-priority tickets

In [20]:
# companies
companies.to_csv('companies.csv', index=False) 
FileLink('companies.csv')

/kaggle/working/companies.csv

In [21]:
# users
users_df.to_csv('users.csv', index=False)
FileLink('users.csv')

/kaggle/working/users.csv

In [22]:
# subscriptions
subscriptions_df.to_csv('subscriptions.csv', index=False)
FileLink('subscriptions.csv')

/kaggle/working/subscriptions.csv

In [23]:
# product usage
product_usage_df.to_csv('product_usage.csv', index=False)
FileLink('product_usage.csv')

/kaggle/working/product_usage.csv

In [24]:
# support tickets
tickets_df.to_csv('support_tickets.csv', index=False)
FileLink('support_tickets.csv')

/kaggle/working/support_tickets.csv